# Laboratory 5 — From Paired Cues to a Motor Response

Companion to Chapters 4 and 10. CS = a visual cue; US = a vibration cue. A bounded trial-based coactivity rule changes the CS connection. A separate physical-unit LIF neuron reads out the learned drive. Ideal motor distance is an explicit controller assumption; this is not a Gazebo or hardware run.

Run all cells, then change one parameter at a time. Figures are synthetic teaching outputs.

[Run in Colab](https://colab.research.google.com/github/an-hongyu/Introduction-to-Neuromorphic-Embodied-AI/blob/main/notebooks/conditioning.ipynb)

## 1. LIF readout (pA, mV, pF, nS, ms)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_lif(current_pa, dt_ms=0.1, capacitance_pf=200.0,
                 leak_ns=10.0, rest_mv=-65.0, threshold_mv=-50.0,
                 reset_mv=-65.0, refractory_ms=2.0, spiking=True):
    """One current value per time interval; return time, voltage, spike times.

    Units: pA, ms, pF, nS, mV. Initial voltage is reset_mv.
    Exact passive update for piecewise-constant current; endpoint events.
    spiking=False exposes the passive RC response without threshold/reset.
    """
    current = np.asarray(current_pa, dtype=float)
    if current.ndim != 1 or current.size == 0 or not np.isfinite(current).all():
        raise ValueError('Supply a nonempty, finite 1-D current array.')
    if dt_ms <= 0 or capacitance_pf <= 0 or leak_ns <= 0 or refractory_ms < 0:
        raise ValueError('dt, capacitance, and leak must be positive; refractory >= 0.')
    if reset_mv >= threshold_mv:
        raise ValueError('Reset must be below threshold.')
    tau_ms = capacitance_pf / leak_ns
    decay = np.exp(-dt_ms / tau_ms)
    time = np.arange(current.size + 1) * dt_ms
    voltage = np.empty(time.size)
    voltage[0] = reset_mv
    spikes, hold = [], 0
    # Round refractory duration upward to a whole number of time steps.
    refractory_steps = int(np.ceil(refractory_ms / dt_ms - 1e-12))
    for k, injected_pa in enumerate(current):
        if hold > 0:
            voltage[k + 1] = reset_mv
            hold -= 1
        else:
            steady_mv = rest_mv + injected_pa / leak_ns
            voltage[k + 1] = steady_mv + (voltage[k] - steady_mv) * decay
            if spiking and voltage[k + 1] >= threshold_mv:
                spikes.append(time[k + 1])
                voltage[k + 1] = reset_mv
                hold = refractory_steps
    return time, voltage, np.asarray(spikes)

def pulse_train(time_ms, starts_ms, width_ms=4.0, amplitude_pa=600.0):
    """Rectangular current pulses; time_ms contains interval start times."""
    current = np.zeros_like(time_ms, dtype=float)
    for start in starts_ms:
        current += amplitude_pa * ((time_ms >= start) & (time_ms < start + width_ms))
    return current

def analytical_rate(current_pa, capacitance_pf=200.0, leak_ns=10.0,
                    rest_mv=-65.0, threshold_mv=-50.0, reset_mv=-65.0,
                    refractory_ms=2.0):
    steady_mv = rest_mv + current_pa / leak_ns
    if steady_mv <= threshold_mv:
        return 0.0
    crossing_ms = (capacitance_pf / leak_ns) * np.log(
        (steady_mv - reset_mv) / (steady_mv - threshold_mv))
    return 1000.0 / (crossing_ms + refractory_ms)

def plot_response(ax, time, voltage, spikes, label, color, threshold=-50.0):
    ax.plot(time, voltage, color=color, label=label, lw=1.8)
    if len(spikes):
        ax.vlines(spikes, threshold + 0.7, threshold + 3.0, color=color, lw=1.5)
    ax.axhline(threshold, color='0.55', ls='--', lw=0.9)
    ax.set(xlabel='Time (ms)', ylabel='Voltage (mV)')

## 2. Bounded coactivity learning and frozen probes

In [ ]:
def train_association(mode='paired', trials=12, eta=.1, initial=.05):
    """Bounded trial-based Hebbian example. x,y,w are dimensionless.
    This is not Oja or STDP, and it does not simulate animal fear.
    """
    weight, history = initial, [initial]
    for _ in range(trials):
        schedule = [(1., 1.)] if mode == 'paired' else [(1., 0.), (0., 1.)]
        for cs, us in schedule:
            post = float(weight * cs + us > .5)
            if mode != 'frozen':
                weight += eta * cs * post * (1 - weight)
        history.append(weight)
    return np.asarray(history)

def probe_association(weight, cs, us):
    # Learning is disabled during every probe. Readout is a separate LIF neuron.
    drive = 50 + 200 * (weight * cs + us)  # pA: declared teaching mapping
    t, v, events = simulate_lif(np.full(5000, drive))
    move = len(events) >= 3  # response criterion over the 500 ms window
    return t, v, events, drive, .1 * 2.0 if move else 0.0  # ideal travel distance, m

## 3. Paired, unpaired, and frozen-learning controls

In [ ]:
# EDIT eta or trials. Compare paired, unpaired, and frozen-learning controls.
trials, eta = 12, .1
histories = {mode: train_association(mode, trials=trials, eta=eta) for mode in ['paired', 'unpaired', 'frozen']}
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
for mode, values in histories.items(): ax[0].plot(values, marker='o', ms=3, label=mode)
ax[0].set(xlabel='Training trial', ylabel='CS connection weight', ylim=(0, 1), title='A. Weight change requires coactivity'); ax[0].legend()
labels = ['CS only', 'US only', 'CS + US']
before = [len(probe_association(.05, *p)[2]) for p in [(1,0), (0,1), (1,1)]]
after = [len(probe_association(histories['paired'][-1], *p)[2]) for p in [(1,0), (0,1), (1,1)]]
x = np.arange(3)
ax[1].bar(x-.18, before, .36, label='Before', color='#b75d45'); ax[1].bar(x+.18, after, .36, label='After paired training', color='#24628a')
ax[1].set(xticks=x, xticklabels=labels, ylabel='Readout spikes / 500 ms', title='B. Frozen-weight probe trials'); ax[1].legend(fontsize=9)
plt.show()

## 4. Separate voltage, event, and motor panels

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 6.5), constrained_layout=True,
                       gridspec_kw={'height_ratios': [3, 1, 1]})
for row, (weight, color, label) in enumerate([(.05, '#b75d45', 'Before'), (histories['paired'][-1], '#24628a', 'After')]):
    t, v, events, drive, distance = probe_association(weight, 1, 0)
    ax[0].plot(t, v, color=color, label=f'{label}: {drive:.1f} pA')
    ax[1].vlines(events, row+.1, row+.8, color=color)
    ax[2].barh(row, distance, color=color, height=.55)
ax[0].axhline(-50, ls='--', color='.6'); ax[0].set(ylabel='Voltage (mV)', title='CS-alone recall: voltage, output events, and ideal motor response'); ax[0].legend(fontsize=9)
ax[1].set(xlabel='Probe time (ms)', ylabel='Events', yticks=[.45,1.45], yticklabels=['Before','After'], xlim=(0,500))
ax[0].set(xlim=(0,500))
ax[2].set(xlabel='Ideal travel AFTER the probe: 0.1 m/s × 2 s (m)', xlim=(0,.24), yticks=[0,1], yticklabels=['Before','After'])
plt.show()
assert np.isclose(histories['paired'][-1], 1 - .95 * (1-eta)**trials)
assert np.allclose(histories['unpaired'], .05) and np.allclose(histories['frozen'], .05)
assert probe_association(.05, 1, 0)[4] == 0
# The default reference must respond; edits may deliberately abolish recall.
assert probe_association(train_association('paired')[-1], 1, 0)[4] == .2
assert probe_association(.05, 0, 1)[4] == .2
print('Association checks passed: paired/unpaired/frozen controls and learning-disabled recall.')
print('Final paired weight:', histories['paired'][-1])

## References

[1] Donald O. Hebb, *The Organization of Behavior: A Neuropsychological Theory*. New York, NY, USA: Wiley, 1949. [Catalog](https://wellcomecollection.org/works/g4sfzum4).

[2] Noah Zins, Yan Zhang, and Hongyu An, “Implementation of Associative Memory Learning in Mobile Robots Using Neuromorphic Computing,” in *Neuromorphic Computing*, 2023, doi: [10.5772/intechopen.110364](https://www.intechopen.com/chapters/86265).

**Model scope.** The teaching model here differs from that publication in sensory processing, learning rule, neuron count, and deployment platform. It illustrates controls and does not reproduce the published experiment.